# LeafGuard - Pipeline completo de classificacao

Este notebook documenta e executa o pipeline completo do LeafGuard: exploracao dos dados, preprocessing, Transfer Learning, treinamento, avaliacao e preparacao da inferencia. O dataset e carregado diretamente do Hugging Face, sem copiar as imagens para o repositorio.

Perguntas principais:
- Quais splits, classes e quantidades existem?
- A distribuicao das classes e equilibrada?
- Qual e o formato das imagens?
- Existem valores ausentes ou amostras que exigem atencao?
- Como o modelo e treinado e avaliado?
- Como o checkpoint e reutilizado pela API?

A execucao requer internet para carregar o dataset e os pacotes listados em `requirements.txt`.

In [ ]:
from collections import Counter
import random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from datasets import load_dataset
from IPython.display import display

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
plt.style.use("seaborn-v0_8-whitegrid")

## Carregamento

O `load_dataset` baixa e armazena o dataset no cache local do Hugging Face. Isso evita versionar imagens no Git e mantém a exploracao reproduzivel.

In [ ]:
DATASET_ID = "AI-Lab-Makerere/beans"
dataset = load_dataset(DATASET_ID)

print(dataset)
print("Splits:", list(dataset.keys()))
for split_name, split in dataset.items():
    print(f"{split_name}: {len(split):,} exemplos")
print("Features do primeiro split:", dataset[next(iter(dataset))].features)

## Classes e distribuicao

A coluna `labels` e uma classe inteira com nomes definidos pelo dataset. Vamos converter os rotulos para nomes legiveis e comparar os splits.

In [ ]:
first_split = dataset[next(iter(dataset))]
label_feature = first_split.features["labels"]
class_names = label_feature.names
print("Classes na ordem do dataset:", class_names)

distribution_rows = []
for split_name, split in dataset.items():
    counts = Counter(split["labels"])
    for label_id, class_name in enumerate(class_names):
        distribution_rows.append({
            "split": split_name,
            "class_id": label_id,
            "class": class_name,
            "count": counts.get(label_id, 0),
            "percentage": 100 * counts.get(label_id, 0) / len(split),
        })

distribution = pd.DataFrame(distribution_rows)
display(distribution.round({"percentage": 2}))

distribution_pivot = distribution.pivot(index="class", columns="split", values="count")
distribution_pivot.plot(kind="bar", figsize=(9, 4), rot=0, title="Exemplos por classe e split")
plt.ylabel("Quantidade")
plt.xlabel("Classe")
plt.tight_layout()
plt.show()

In [ ]:
print("Valores ausentes por coluna e split:")
missing_rows = []
for split_name, split in dataset.items():
    frame = split.to_pandas()
    for column in frame.columns:
        missing_rows.append({
            "split": split_name,
            "column": column,
            "missing": int(frame[column].isna().sum()),
        })
missing = pd.DataFrame(missing_rows)
display(missing)

label_values = sorted(set(value for split in dataset.values() for value in split["labels"]))
print("Rotulos observados:", label_values)
print("Quantidade de rotulos esperada:", len(class_names))

## Dimensoes, modos e exemplos

As imagens podem ter dimensoes variadas. A amostragem abaixo mede largura, altura e modo de cor para todos os itens, sem carregar uma copia adicional dos arquivos no repositorio.

In [ ]:
image_rows = []
for split_name, split in dataset.items():
    for index, image in enumerate(split["image"]):
        image_rows.append({
            "split": split_name,
            "index": index,
            "width": image.width,
            "height": image.height,
            "mode": image.mode,
            "label": class_names[split["labels"][index]],
        })
image_stats = pd.DataFrame(image_rows)
display(image_stats.head())
print("Modos de cor:")
display(image_stats["mode"].value_counts().rename_axis("mode").to_frame("count"))
print("Resumo das dimensoes por split:")
display(image_stats.groupby("split")[["width", "height"]].describe().round(1))
print("Dimensoes mais frequentes:")
display(image_stats.groupby(["width", "height"]).size().sort_values(ascending=False).head(10).rename("count").to_frame())

In [ ]:
rng = np.random.default_rng(SEED)
fig, axes = plt.subplots(len(dataset), 4, figsize=(12, 3 * len(dataset)))
axes = np.atleast_2d(axes)

for row, (split_name, split) in enumerate(dataset.items()):
    sample_size = min(4, len(split))
    indices = rng.choice(len(split), size=sample_size, replace=False)
    for column, index in enumerate(indices):
        item = split[int(index)]
        axes[row, column].imshow(item["image"])
        axes[row, column].set_title(f"{split_name} | {class_names[item['labels']]}")
        axes[row, column].axis("off")
    for column in range(sample_size, 4):
        axes[row, column].axis("off")

plt.suptitle("Amostras aleatorias por split", y=1.02, fontsize=14)
plt.tight_layout()
plt.show()

## Checklist de problemas e decisoes preliminares

A celula seguinte transforma os resultados observados em um resumo inicial. Ela nao substitui a analise visual: registre aqui qualquer amostra corrompida, rotulo suspeito ou diferenca relevante entre splits encontrada durante a execucao.

Decisoes esperadas para a proxima etapa, a confirmar apos esta exploracao:
- redimensionar todas as imagens para um tamanho fixo compativel com MobileNetV3-Small;
- converter imagens para RGB para garantir tres canais;
- normalizar usando as estatisticas esperadas pelos pesos pre-treinados;
- aplicar augmentation somente no treino;
- preservar validacao e teste com transformacoes deterministicas;
- considerar pesos por classe ou estratificacao somente se a distribuicao observada justificar.

In [ ]:
print("Resumo automatico da exploracao")
print(f"- Splits: {', '.join(dataset.keys())}")
print(f"- Classes ({len(class_names)}): {', '.join(class_names)}")
print(f"- Total de imagens: {sum(len(split) for split in dataset.values()):,}")
print(f"- Modos de imagem: {image_stats['mode'].value_counts().to_dict()}")
print(f"- Dimensoes unicas: {image_stats[['width', 'height']].drop_duplicates().shape[0]} combinacao(oes)")
print(f"- Valores ausentes encontrados: {int(missing['missing'].sum())}")

if int(missing["missing"].sum()) == 0 and set(image_stats["mode"]) == {"RGB"}:
    print("- Nenhum valor ausente foi encontrado e todas as imagens ja estao em RGB.")
else:
    print("- Revise a tabela de ausentes e modos antes de definir o preprocessing.")

print()
print("Proximo passo: validar estes resultados e somente depois implementar preprocessing.")

## Conclusoes da Etapa 1

- O dataset possui 1.295 imagens: 1.034 para treino, 133 para validacao e 128 para teste.
- As tres classes estao aproximadamente equilibradas em todos os splits, com cerca de 33% cada. Nao ha indicacao, nesta etapa, para usar pesos de classe.
- Todas as imagens sao RGB e possuem dimensao fixa de 500 x 500 pixels.
- Nao foram encontrados valores ausentes, rotulos fora do conjunto esperado ou imagens em modo de cor diferente.
- As amostras apresentam variacao de iluminacao, fundo, enquadramento e estado visual das folhas. Essa variacao justifica augmentation moderado no treino, mas nao permite concluir ainda quais transformacoes serao mais adequadas.

### Decisoes para o preprocessing

Para a proxima etapa, faz sentido redimensionar as imagens para o tamanho esperado pelo backbone leve escolhido, converter explicitamente para RGB e normalizar com as estatisticas dos pesos pre-treinados. O augmentation deve ser aplicado somente ao treino; validacao e teste devem usar transformacoes deterministicas. O batch size sera definido de forma conservadora e ajustavel, considerando a execucao em CPU ou GPU gratuita.

Nenhum treinamento, modelo ou API foi implementado nesta etapa.

## Etapa 2 — Preprocessing e DataLoaders

O módulo `src.data` fornece transformações e DataLoaders reutilizáveis:
- Redimensionamento para 224 x 224 (tamanho esperado por MobileNetV3).
- Normalização usando médias e desvios ImageNet.
- Augmentation somente no treino: flip horizontal, rotação, color jitter.
- Validação e teste com transformações determinísticas.
- Batch size configurável.
- Uso de CPU e GPU transparente.

A célula abaixo valida que o módulo funciona e confirma os shapes e tipos dos batches.

In [ ]:
import sys
from pathlib import Path

project_root = Path.cwd()
if not (project_root / "src").exists():
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

from src.data import DataConfig, create_dataloaders

# Criar DataLoaders usando o dataset carregado anteriormente
config = DataConfig(image_size=224, batch_size=16)
loaders = create_dataloaders(dataset, config)

print("Resumo dos DataLoaders:")
for split_name, loader in loaders.items():
    print(f"  {split_name}: {len(loader.dataset)} exemplos, {len(loader)} batches")

# Examinar um batch de treino
images_train, labels_train = next(iter(loaders["train"]))
print(f"\nBatch de treino:")
print(f"  Imagens: shape={images_train.shape}, dtype={images_train.dtype}")
print(f"  Rotulos: shape={labels_train.shape}, dtype={labels_train.dtype}")
print(f"  Min/Max de pixels: {images_train.min():.3f} / {images_train.max():.3f}")

# Examinar um batch de validacao
images_val, labels_val = next(iter(loaders["validation"]))
print(f"\nBatch de validacao:")
print(f"  Imagens: shape={images_val.shape}, dtype={images_val.dtype}")
print(f"  Rotulos: shape={labels_val.shape}, dtype={labels_val.dtype}")

In [ ]:
# Validar que o treino tem augmentation aleatória e validação é determinística
import torch

print("Verificacao 1: Augmentation no treino (imagens diferentes a cada iteracao)")
img1_train, _ = next(iter(loaders["train"]))
img1_train_again, _ = next(iter(loaders["train"]))
# Se a augmentation funciona, as imagens do mesmo exemplo devem ser diferentes
# (ao menos um pixel deve variar). Usamos o mesmo índice para confirmar variação.
are_different = not torch.allclose(img1_train, img1_train_again)
print(f"  Imagens do treino variam entre iteracoes: {are_different}")

print("\nVerificacao 2: Validacao e teste sem randomness (mesmos dados, mesma ordem)")
val_iter1 = iter(loaders["validation"])
img_val_1, lbl_val_1 = next(val_iter1)

val_iter2 = iter(loaders["validation"])
img_val_2, lbl_val_2 = next(val_iter2)

are_identical_val = torch.allclose(img_val_1, img_val_2) and torch.equal(lbl_val_1, lbl_val_2)
print(f"  Primeiro batch de validacao idêntico após novo iterador: {are_identical_val}")

print("\nResumo:")
print(f"- Reshape: 500x500 -> 224x224 (MobileNetV3)")
print("- Normalizacao: estatisticas ImageNet")
print(f"- Treino: augmentation aleatório (flip, rotacao, color jitter)")
print(f"- Validacao/teste: determinístico")
print(f"- Batch size: {config.batch_size}, Num workers: {config.num_workers}")
print(f"- Próximo passo: Etapa 3 — Carregamento do modelo MobileNetV3 e configuracao de Transfer Learning")

## Etapa 3 — Modelo e Transfer Learning

### Motivacao

O dataset possui apenas 1.034 exemplos de treino, insuficiente para treinar uma CNN do zero sem overfitting massivo. **Transfer Learning** resolve isso reutilizando pesos pré-treinados em ImageNet (14 milhões de imagens):

1. **Backbone (Feature Extractor)**: Pesos ImageNet congelados, extraem padrões visuais gerais (bordas, texturas, formas).
2. **Classifier (Cabeça)**: Camadas novas, treináveis, aprendem mapeamentos específicos para bean diseases.

Isso permite treinar com menos dados e menos poder computacional.

### Arquitetura

Usamos **MobileNetV3-Small**:
- Projetado para dispositivos móveis e recursos limitados.
- ~2.5M parâmetros (vs 25M+ em ResNet-50).
- Compatível com CPU e GPUs gratuitas.
- Pré-treinado em ImageNet.

O módulo `src/model.py` fornece funções para:
- Carregar backbone com pesos congelados.
- Substituir classificador por uma cabeça de 3 classes.
- Contar parâmetros treináveis.
- Descongelar camadas para fine-tuning posterior (opcional).

In [ ]:
import importlib
import src.model
importlib.reload(src.model)

from src.model import ModelConfig, create_model, freeze_backbone, count_parameters

# Criar modelo com backbone pre-treinado no ImageNet.
config_model = ModelConfig(num_classes=3)
model = create_model(config_model, pretrained=True)

print("Arquitetura do classificador:")
print(model.classifier)

print("\nContagem de parametros:")
params = count_parameters(model)
print(f"  Total: {params['total']:,}")
print(f"  Treináveis: {params['trainable']:,}")
print(f"  Congelados (backbone): {params['frozen']:,}")
print(f"  Percentual treinável: {100 * params['trainable'] / params['total']:.1f}%")

print("\nStatus do backbone (congelado):")
backbone_frozen = all(
    not p.requires_grad
    for name, p in model.named_parameters()
    if "classifier" not in name
)
print(f"  Backbone congelado: {backbone_frozen}")

# Verificar dispositivo suportado
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)
print(f"\nDispositivo: {device}")

print("\nOs pesos ImageNet foram carregados; a cabeca classificadora sera adaptada as 3 classes.")
print("Proxima etapa: executar o treinamento e acompanhar a validacao.")

In [ ]:
# Demonstrar congelamento de backbone após carregamento
freeze_backbone(model, freeze=True)

print("Após congelar backbone:")
params_frozen = count_parameters(model)
print(f"  Total: {params_frozen['total']:,}")
print(f"  Treináveis: {params_frozen['trainable']:,}")
print(f"  Congelados: {params_frozen['frozen']:,}")
print(f"  Percentual treinável: {100 * params_frozen['trainable'] / params_frozen['total']:.1f}%")

print("\nFluxo esperado no treinamento:")
print("  1. Carregar modelo com pesos ImageNet (primeira execução).")
print("  2. Congelar backbone (feature extractor).")
print("  3. Treinar apenas classificador (cabeça nova).")
print("  4. Após 5-10 épocas, avaliar e considerar fine-tuning de camadas próximas.")
print("\nIsto é Transfer Learning: aproveita conhecimento ImageNet, treina pouco.")

## Etapa 4 — Treinamento e validacao

O modulo `src.train` fornece os loops reutilizaveis:
- `train_epoch()`: forward pass, Cross-Entropy, backpropagation e atualizacao do otimizador.
- `evaluate()`: avaliacao sem gradientes no conjunto de validacao ou teste.

Primeiro executamos um proof-of-concept curto com pesos aleatorios para validar o pipeline. Em seguida, a celula de treinamento completo usa o backbone pre-treinado no ImageNet, congela suas camadas e salva o melhor checkpoint conforme a accuracy de validacao.

In [ ]:
import torch.nn as nn
import importlib
import src.train
importlib.reload(src.train)

from src.train import TrainConfig, train_epoch, evaluate

# Preparar modelo novo com Transfer Learning
model_train = create_model(config_model, pretrained=False)
freeze_backbone(model_train, freeze=True)
model_train = model_train.to(device)

# Configuracao de treinamento
train_config = TrainConfig(learning_rate=0.001, num_epochs=3, device=device)
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model_train.parameters(), lr=train_config.learning_rate)

print("Iniciando proof-of-concept de treinamento (3 épocas)...")
print(f"Device: {device}, Learning rate: {train_config.learning_rate}\n")

history = {"train": [], "val": []}

for epoch in range(train_config.num_epochs):
    train_metrics = train_epoch(model_train, loaders["train"], loss_fn, optimizer, device)
    val_metrics = evaluate(model_train, loaders["validation"], loss_fn, device)

    history["train"].append(train_metrics)
    history["val"].append(val_metrics)

    print(f"Epoch {epoch + 1}/{train_config.num_epochs}:")
    print(f"  Train - Loss: {train_metrics['loss']:.4f}, Accuracy: {train_metrics['accuracy']:.4f}")
    print(f"  Val   - Loss: {val_metrics['loss']:.4f}, Accuracy: {val_metrics['accuracy']:.4f}\n")

print("Proof-of-concept concluído! Pipeline de treinamento validado.")

In [ ]:
# Treinamento completo com pesos ImageNet
from pathlib import Path
import copy

print("Iniciando treinamento completo com pesos ImageNet...")

model_full = create_model(config_model, pretrained=True)
freeze_backbone(model_full, freeze=True)
model_full = model_full.to(device)

full_train_config = TrainConfig(learning_rate=0.001, num_epochs=15, device=device)
full_loss_fn = nn.CrossEntropyLoss()
full_optimizer = torch.optim.Adam(
    [parameter for parameter in model_full.parameters() if parameter.requires_grad],
    lr=full_train_config.learning_rate,
)

history_full = {"train": [], "val": []}
best_val_accuracy = -1.0
best_state = None

for epoch in range(full_train_config.num_epochs):
    train_metrics_full = train_epoch(
        model_full,
        loaders["train"],
        full_loss_fn,
        full_optimizer,
        device,
    )
    val_metrics_full = evaluate(
        model_full,
        loaders["validation"],
        full_loss_fn,
        device,
    )

    history_full["train"].append(train_metrics_full)
    history_full["val"].append(val_metrics_full)

    if val_metrics_full["accuracy"] > best_val_accuracy:
        best_val_accuracy = val_metrics_full["accuracy"]
        best_state = copy.deepcopy(model_full.state_dict())

    print(
        f"Epoch {epoch + 1:02d}/{full_train_config.num_epochs} | "
        f"train loss={train_metrics_full['loss']:.4f}, "
        f"train acc={train_metrics_full['accuracy']:.4f} | "
        f"val loss={val_metrics_full['loss']:.4f}, "
        f"val acc={val_metrics_full['accuracy']:.4f}"
    )

if best_state is None:
    raise RuntimeError("Nenhum checkpoint de validação foi produzido.")

model_full.load_state_dict(best_state)
checkpoint_dir = Path("../artifacts")
checkpoint_dir.mkdir(parents=True, exist_ok=True)
checkpoint_path = checkpoint_dir / "mobilenetv3_beans_best.pt"
torch.save(
    {
        "model_state_dict": model_full.state_dict(),
        "class_names": class_names,
        "model_config": config_model,
        "best_val_accuracy": best_val_accuracy,
        "history": history_full,
    },
    checkpoint_path,
)

model_train = model_full
history = history_full
print(f"\nMelhor accuracy de validacao: {best_val_accuracy:.4f}")
print(f"Checkpoint salvo em: {checkpoint_path}")

## Etapa 5 — Avaliação Detalhada

O módulo `src/evaluate.py` fornece funções para avaliar o modelo com:
- Accuracy, Precision, Recall, F1-Score
- Métricas por classe
- Matriz de confusão

A célula abaixo avalia o modelo treinado acima no conjunto de teste e exibe as métricas detalhadas.

In [ ]:
%pip install scikit-learn -q

import importlib
import src.evaluate
importlib.reload(src.evaluate)

from src.evaluate import evaluate_detailed, print_metrics

# Avaliar no conjunto de teste usando o melhor checkpoint ImageNet
# (model_train foi substituido pelo melhor estado no treinamento completo.)
test_metrics = evaluate_detailed(model_train, loaders["test"], device, class_names)

print("Avaliacao no conjunto de TESTE:\n")
print_metrics(test_metrics, class_names)

print("\n" + "="*60)
print("Notas sobre os resultados:")
print("="*60)
print("1. Este modelo foi treinado com pesos ImageNet e o melhor checkpoint de validacao.")
print("2. A matriz de confusao mostra quais classes o modelo confunde.")
print("3. Precision vs Recall trade-off: importante considerar para o caso de uso.")
print("4. F1-Score e a media harmonica: bom para avaliar desempenho geral.")

## Resumo completo do projeto

Este notebook documenta as etapas experimentais do LeafGuard. O codigo reutilizavel de producao fica em `src/`, a API em `api/` e os testes em `tests/`.

### Etapas 1-5 — Dados, modelo, treino e avaliacao

- **Etapa 1:** dataset Beans explorado: 1.295 imagens, tres classes equilibradas, imagens RGB de 500 x 500 e sem valores ausentes.
- **Etapa 2:** imagens redimensionadas para 224 x 224, normalizadas com estatisticas ImageNet e augmentadas somente no treino.
- **Etapa 3:** MobileNetV3-Small com backbone ImageNet e cabeca `576 -> 256 -> 3`.
- **Etapa 4:** backbone congelado e classificador treinado com Adam e Cross-Entropy.
- **Etapa 5:** accuracy, precision, recall, F1 e matriz de confusao calculados no teste.

### Resultado do treinamento completo

- Melhor accuracy de validacao: **90,98%** na epoca 11.
- Accuracy no teste: **87,50%**.
- F1 ponderado no teste: **87,60%**.
- Checkpoint: `artifacts/mobilenetv3_beans_best.pt`.

### Etapas 6-7 — Inferencia e API

- `src/predict.py` carrega o checkpoint, aplica o preprocessing deterministico e retorna classe, confianca e probabilidades.
- `api/main.py` expoe `GET /health` e `POST /predict` com upload multipart.
- O modelo e mantido em cache para evitar recarregamento a cada requisicao.

### Etapa 8 — Docker

- `Dockerfile` empacota a API, as dependencias e o checkpoint.
- A imagem `leafguard-api` foi construida com sucesso.

### Etapas 9-10 — Testes e documentacao

- Os testes em `tests/` cobrem inferencia e endpoints da API.
- Resultado atual: **4 testes aprovados**.
- O uso local, Docker, resultados, limitacoes e contrato da API estao documentados em `README.md`.

### Ordem recomendada de execucao

1. Execute as celulas de exploracao e preprocessing.
2. Execute o proof-of-concept para validar o pipeline.
3. Execute o treinamento completo para gerar ou atualizar o checkpoint.
4. Execute a avaliacao final no conjunto de teste.
5. Use `pytest -q` para validar o codigo da API e da inferencia.
6. Inicie a API com Uvicorn ou pelo container Docker.

O notebook e um registro didatico e experimental. O fluxo de servico nao depende dele: a API utiliza diretamente `src/predict.py` e o checkpoint salvo em `artifacts/`.